# Public source archive

Recovered historical code, prepared for public review. AI assisted the original coding and this archive preparation; team projects retain team attribution in the project README.

Outputs, notebook attachments, personal metadata, and embedded organizer datasets have been removed. Narrative cells retain headings only. Supply authorized input files before running data-dependent notebooks. This archive does not claim that manual checks were completed or that the code was independently developed.


# League of Page｜Celonis团队共用分析版
### 第一次使用，只做这4步


## 第1步｜准备相同的数据和运行目录


In [ ]:
from pathlib import Path
import os
if '_TEAM_ROOT' not in globals():
    _TEAM_ROOT = (Path('/content') if Path('/content').is_dir() else Path.cwd()) / 'Celonis_Team_Workspace'
_TEAM_ROOT.mkdir(parents=True,exist_ok=True)
os.chdir(_TEAM_ROOT)
_team_logs = []
# Public archive: original embedded organizer data removed.
from pathlib import Path
input_dir = Path('league_of_page_data')
required_files = ['organizations.csv', 'job_postings.csv', 'reviews.csv',
                  'review_job_role_links.csv', 'data_dictionary.csv']
missing_files = [name for name in required_files if not (input_dir/name).is_file()]
if missing_files:
    raise FileNotFoundError(
        'Place your authorized source CSVs in ' + str(input_dir.resolve())
        + '. Missing: ' + ', '.join(missing_files)
    )


## 第2步｜读取原始表，确认分析对象


In [ ]:
import io, contextlib
import matplotlib.pyplot as plt
_step_log = io.StringIO()
_previous_show = plt.show
plt.show = lambda: plt.close('all')
try:
    with contextlib.redirect_stdout(_step_log):
        """Reproducible, non-destructive preprocessing for the selected three employers.
        Run beside the five organizer CSV files. Only derived columns/files are written.
        """
        from pathlib import Path
        import csv, json, re, hashlib, platform, sys
        import pandas as pd

        NAMES = ['Celonis', 'Anaplan', 'AlphaSense']
        FILES = ['organizations.csv', 'job_postings.csv', 'reviews.csv',
                 'review_job_role_links.csv', 'data_dictionary.csv']
        DATA = Path('league_of_page_data')
        OUTPUT = Path('preprocess_outputs')
        OUTPUT.mkdir(parents=True, exist_ok=True)

        def locate(name):
            if (DATA/name).is_file():
                return DATA/name
            candidates = sorted(DATA.glob('*-' + name))
            return candidates[0] if len(candidates) == 1 else None

        if any(locate(n) is None for n in FILES):
            try:
                from google.colab import files
                print('Upload the five original CSV files together.')
                files.upload()
            except ImportError:
                pass
        paths = {name: locate(name) for name in FILES}
        assert all(paths.values()), 'Missing original CSV files; place them beside this script/notebook.'


        # Read ALL source fields as strings. Do not convert literal NA/None text to missing.
        source = {name: pd.read_csv(p, dtype=str, keep_default_na=False) for name,p in paths.items()}
        organizations = source['organizations.csv']
        organizations = organizations[organizations.organization_name.isin(NAMES)].copy()
        assert organizations.organization_name.nunique() == 3
        assert organizations.organization_id.is_unique
        ids = set(organizations.organization_id)
        jobs = source['job_postings.csv']
        jobs = jobs[jobs.organization_id.isin(ids)].copy()
        reviews = source['reviews.csv']
        reviews = reviews[reviews.organization_id.isin(ids)].copy()
        links = source['review_job_role_links.csv']
        links = links[links.organization_id.isin(ids)].copy()
        assert jobs.job_id.is_unique and reviews.review_id.is_unique
        assert jobs.organization_name.isin(NAMES).all() and reviews.organization_name.isin(NAMES).all()

        def clean_text(value):
            # Preserve spelling, language and negation. Keep all original fields untouched.
            return re.sub(r'[ \t\u00a0]+', ' ', str(value).replace('\r\n','\n').replace('\r','\n')).strip()

        def flag_missing(value):
            return not str(value).strip()

        def nullable_boolean(series):
            return series.str.strip().str.lower().map({'true':True, 'false':False}).astype('boolean')


finally:
    plt.show = _previous_show
_team_logs.append(_step_log.getvalue())

## 第3步｜预处理招聘广告


In [ ]:
import io, contextlib
import matplotlib.pyplot as plt
_step_log = io.StringIO()
_previous_show = plt.show
plt.show = lambda: plt.close('all')
try:
    with contextlib.redirect_stdout(_step_log):
        for col in ['title','description','location_text']:
            jobs[col+'_clean'] = jobs[col].map(clean_text)
        jobs['description_char_count'] = jobs.description_clean.str.len()
        jobs['short_description_flag'] = jobs.description_char_count.lt(1500)
        jobs['missing_description_flag'] = jobs.description_clean.eq('')
        jobs['missing_country_code_flag'] = jobs.countryiso.map(flag_missing)
        jobs['early_career_title_flag'] = nullable_boolean(jobs.early_career_title_signal)
        jobs['remote_status_raw'] = jobs.remote_status
        jobs['remote_status_normalized'] = jobs.remote_status.str.strip().str.lower().replace('', 'unknown')
        jobs['remote_status_requires_context_check'] = True
        jobs['posting_datetime_utc'] = pd.to_datetime(jobs.posted_at, utc=True, errors='coerce',format='mixed')
        jobs['posting_date_source_local'] = jobs.posted_at.str.extract(r'^(\d{4}-\d{2}-\d{2})',expand=False)
        jobs['invalid_posting_date_flag'] = jobs.posted_at.ne('') & jobs.posting_datetime_utc.isna()
        jobs['repeated_description_flag'] = jobs.duplicated(['organization_id','text_sha256'],keep=False)
        jobs['description_occurrences_within_employer'] = jobs.groupby(['organization_id','text_sha256']).job_id.transform('size')
        jobs['opening_key_duplicate_flag'] = jobs.duplicated(['organization_id','opening_dedup_key'],keep=False)


finally:
    plt.show = _previous_show
_team_logs.append(_step_log.getvalue())

## 第4步｜预处理员工评价和评分


In [ ]:
import io, contextlib
import matplotlib.pyplot as plt
_step_log = io.StringIO()
_previous_show = plt.show
plt.show = lambda: plt.close('all')
try:
    with contextlib.redirect_stdout(_step_log):
        for col in ['reviewer_job_title','reviewer_location','summary','pros','cons','advice_to_management']:
            reviews[col+'_clean'] = reviews[col].map(clean_text)
        reviews['review_text_clean'] = (
            'SUMMARY: '+reviews.summary_clean+'\nPROS: '+reviews.pros_clean+
            '\nCONS: '+reviews.cons_clean+'\nADVICE: '+reviews.advice_to_management_clean)
        reviews['missing_review_role_flag'] = reviews.reviewer_job_title_clean.eq('')
        reviews['missing_review_location_flag'] = reviews.reviewer_location_clean.eq('')
        reviews['review_date_parsed'] = pd.to_datetime(reviews.review_date,format='%Y-%m-%d',errors='coerce')
        reviews['invalid_review_date_flag'] = reviews.review_date.ne('') & reviews.review_date_parsed.isna()
        reviews['early_career_review_title_flag'] = reviews.reviewer_job_title_clean.str.contains(
            r'\bintern\b|internship|\bjunior\b|graduate|trainee',case=False,regex=True)
        reviews['interview_experience_candidate_flag'] = reviews.review_text_clean.str.contains(
            r'interview process|recently interviewed|challenge interview|application process',case=False,regex=True)
        reviews['repeated_review_text_flag'] = reviews.duplicated(['organization_id','text_sha256'],keep=False)

        quality_issues = []
        rating_values = pd.to_numeric(reviews.rating,errors='coerce')
        reviews['rating_valid'] = rating_values.where(rating_values.between(1,5))
        reviews['invalid_overall_rating_flag'] = reviews.rating.ne('') & ~rating_values.between(1,5)
        DIMENSIONS = ['career_opportunities','work_life_balance','compensation_and_benefits']

        def parse_subrating(row):
            if not row.subratings.strip():
                return {}
            try:
                value = json.loads(row.subratings)
                if not isinstance(value,dict):
                    raise ValueError('Expected an object')
                return value
            except (json.JSONDecodeError,ValueError,TypeError):
                quality_issues.append({'source':'reviews','source_id':row.review_id,'field':'subratings',
                  'source_value':row.subratings,'issue':'invalid JSON/object; retained source and left derived scores missing'})
                return {}

        parsed = reviews.apply(parse_subrating,axis=1)
        for dim in DIMENSIONS:
            source_value = parsed.map(lambda x:x.get(dim,''))
            number = pd.to_numeric(source_value,errors='coerce')
            valid = number.between(1,5)
            reviews[dim+'_source_value'] = source_value
            reviews[dim+'_valid'] = number.where(valid)
            reviews[dim+'_invalid_or_missing_flag'] = ~valid
            for i in reviews.index[~valid]:
                v = source_value.loc[i]
                issue = 'missing' if v=='' else ('zero outside valid 1–5 range' if number.loc[i]==0 else 'nonnumeric or outside valid 1–5 range')
                quality_issues.append({'source':'reviews','source_id':reviews.loc[i,'review_id'],
                  'field':dim,'source_value':v,'issue':issue})


finally:
    plt.show = _previous_show
_team_logs.append(_step_log.getvalue())

## 第5步｜检查关联关系，导出清洗结果


In [ ]:
import io, contextlib
import matplotlib.pyplot as plt
_step_log = io.StringIO()
_previous_show = plt.show
plt.show = lambda: plt.close('all')
try:
    with contextlib.redirect_stdout(_step_log):
        # Links are preserved as comparable-role links, never vacancy matches.
        name_lookup = organizations.set_index('organization_id').organization_name
        links['organization_name'] = links.organization_id.map(name_lookup)
        links['within_180_days_flag'] = nullable_boolean(links.within_180_days)
        links['date_gap_days_numeric'] = pd.to_numeric(links.posting_minus_review_days,errors='coerce')
        links['exact_location_text_flag'] = links.location_match.eq('exact_location_text')
        links['confirmed_same_vacancy_flag'] = False
        links['review_id_exists_flag'] = links.review_id.isin(set(reviews.review_id))
        links['job_id_exists_flag'] = links.job_id.isin(set(jobs.job_id))
        assert links.review_id_exists_flag.all() and links.job_id_exists_flag.all(), 'Broken link source ID'
        review_employer = reviews.set_index('review_id').organization_id
        job_employer = jobs.set_index('job_id').organization_id
        assert links.review_id.map(review_employer).eq(links.organization_id).all()
        assert links.job_id.map(job_employer).eq(links.organization_id).all()
        gap_known=links.date_gap_days_numeric.notna() & links.within_180_days_flag.notna()
        assert links.loc[gap_known,'within_180_days_flag'].eq(links.loc[gap_known,'date_gap_days_numeric'].abs().le(180)).all()


        # Retain all records; flags support later justified exclusions or sensitivity checks.
        summary = []
        missing_rows = []
        for name in NAMES:
            a = jobs[jobs.organization_name.eq(name)]
            b = reviews[reviews.organization_name.eq(name)]
            c = links[links.organization_name.eq(name)]
            meta=organizations[organizations.organization_name.eq(name)].iloc[0]
            assert len(a)==int(meta.job_rows) and len(b)==int(meta.review_rows) and len(c)==int(meta.role_link_rows)
            summary.append({'company':name,'jobs':len(a),'reviews':len(b),'role_links':len(c),
              'early_job_title_signals':int(a.early_career_title_flag.sum()),
              'early_review_title_signals':int(b.early_career_review_title_flag.sum()),
              'interview_candidates_in_early_reviews':int((b.early_career_review_title_flag & b.interview_experience_candidate_flag).sum()),
              'review_start':b.review_date.min(),'review_end':b.review_date.max(),
              'missing_country_codes':int(a.missing_country_code_flag.sum()),
              'unique_job_description_texts':a.text_sha256.nunique(),
              'invalid_posting_dates':int(a.invalid_posting_date_flag.sum()),
              'invalid_review_dates':int(b.invalid_review_date_flag.sum())})
            for source_name,raw in [('jobs',source['job_postings.csv']),('reviews',source['reviews.csv'])]:
                group=raw[raw.organization_id.eq(meta.organization_id)]
                for field in group:
                    n=int(group[field].map(flag_missing).sum())
                    missing_rows.append({'company':name,'source':source_name,'field':field,
                      'missing_n':n,'n_records':len(group),'missing_pct':100*n/len(group)})
        summary=pd.DataFrame(summary)


        # Add concise definitions for derived fields, with original dictionary alongside.
        definitions = []
        for table,df,original in [('jobs',jobs,source['job_postings.csv']),('reviews',reviews,source['reviews.csv']),('links',links,source['review_job_role_links.csv'])]:
            for col in df.columns:
                if col not in original.columns:
                    if col.endswith('_clean'): definition='Whitespace-normalized copy. Original text retained. Not translated or sentiment classified.'
                    elif col.endswith('_valid'): definition='Numeric score in valid 1–5 range; invalid/zero/missing source values become missing in this derived field.'
                    elif col.endswith('_source_value'): definition='Value extracted from original subratings JSON before numeric validation.'
                    elif col=='early_career_title_flag': definition='Nullable Boolean copy of organizer title heuristic; not verified eligibility.'
                    elif col=='early_career_review_title_flag': definition='Reviewer title contains intern/internship/junior/graduate/trainee expression; not verified tenure or employment.'
                    elif col=='interview_experience_candidate_flag': definition='Narrow English keyword flag for possible interview/application content. Human checking required; unflagged is not guaranteed workplace experience.'
                    elif col=='confirmed_same_vacancy_flag': definition='Always False. No shared vacancy identifier establishes the same opening.'
                    elif col=='remote_status_requires_context_check': definition='Always True. Read the role-specific description before interpreting stored work-arrangement labels.'
                    elif col=='posting_datetime_utc': definition='Parsed timezone-aware source timestamp normalized to UTC; original timestamp and local date retained.'
                    elif col=='posting_date_source_local': definition='Calendar date explicitly written in source timestamp, before UTC conversion.'
                    elif col=='repeated_description_flag': definition='All members of an employer+description-hash group with >1 advertisement; retained, not automatically dropped.'
                    elif col=='description_occurrences_within_employer': definition='Advertisement count sharing the same employer and source description hash; not a vacancy count.'
                    elif col=='short_description_flag': definition='Normalized description has <1500 characters. Screening threshold only; not confirmed truncation.'
                    elif col=='description_char_count': definition='Character count of whitespace-normalized description.'
                    elif col=='remote_status_raw': definition='Untouched copy of source remote_status.'
                    elif col=='remote_status_normalized': definition='Lowercase/trimmed work arrangement label; blank becomes unknown; no semantic correction.'
                    elif col=='opening_key_duplicate_flag': definition='Repeated organization_id + organizer opening_dedup_key; retained; not proven same vacancy.'
                    elif col=='repeated_review_text_flag': definition='Repeated organization_id + review text hash; retained for later context checks.'
                    elif col=='within_180_days_flag': definition='Nullable Boolean copy of organizer linkage flag, checked against absolute numeric day gap.'
                    elif col=='date_gap_days_numeric': definition='Numeric copy of posting_minus_review_days.'
                    elif col=='exact_location_text_flag': definition='True only if source location_match is exact_location_text; not verified employment.'
                    elif col=='organization_name': definition='Employer display name mapped from organizations by organization_id.'
                    elif col.endswith('_id_exists_flag'): definition='Whether linked ID is present in the selected source; required True in this preprocessing.'
                    elif 'missing' in col or 'invalid' in col: definition='Quality flag; inspect the field name and raw source. Original record retained.'
                    elif col=='review_date_parsed': definition='Parsed source review calendar date; invalid values become missing.'
                    else: definition='Derived field. See executable preprocessing code for the exact rule.'
                    definitions.append({'table':table,'derived_field':col,'definition':definition})

        output_tables = {'organizations_selected.csv':organizations,'job_postings_preprocessed.csv':jobs,
         'reviews_preprocessed.csv':reviews,'role_links_preprocessed.csv':links,'sample_summary.csv':summary,
         'missingness_by_company.csv':pd.DataFrame(missing_rows),'subrating_quality_log.csv':pd.DataFrame(quality_issues),
         'derived_field_dictionary.csv':pd.DataFrame(definitions),'original_data_dictionary.csv':source['data_dictionary.csv']}
        for name,df in output_tables.items():
            df.to_csv(OUTPUT/name,index=False,encoding='utf-8-sig',date_format='%Y-%m-%dT%H:%M:%S%z')
        jobs[jobs.early_career_title_flag.fillna(False)].to_csv(OUTPUT/'early_title_jobs_CHECK_ELIGIBILITY.csv',index=False,encoding='utf-8-sig')
        reviews[reviews.interview_experience_candidate_flag].to_csv(OUTPUT/'interview_candidates_CHECK_CONTEXT.csv',index=False,encoding='utf-8-sig')
        manifest = {'companies':NAMES,'python':platform.python_version(),'pandas':pd.__version__,
         'inputs':{name:{'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'bytes':path.stat().st_size} for name,path in paths.items()},
         'policy':'No source fields overwritten; no selected rows dropped; no Cartesian join; no geography or currency guessed.',
         'counts':{'jobs':len(jobs),'reviews':len(reviews),'links':len(links)},
         'human_validation':'Pending; preprocessing checks are not completed organizer manual text validation.'}
        (OUTPUT/'preprocessing_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))


        # Verify saved values, source preservation and primary/foreign keys.
        for output_name,df in output_tables.items():
            check=pd.read_csv(OUTPUT/output_name,dtype=str,keep_default_na=False)
            assert len(check)==len(df), f'Export count mismatch: {output_name}'
        for selected,raw_name,key,out_name in [(jobs,'job_postings.csv','job_id','job_postings_preprocessed.csv'),(reviews,'reviews.csv','review_id','reviews_preprocessed.csv')]:
            raw=source[raw_name].set_index(key)
            exported=pd.read_csv(OUTPUT/out_name,dtype=str,keep_default_na=False).set_index(key)
            for field in raw.columns:
                assert exported[field].eq(raw.loc[exported.index,field]).all(), f'Original field changed: {out_name}/{field}'
        assert len(jobs)==758 and len(reviews)==933 and len(links)==66
        print(summary.to_string(index=False))
        print('\nPASS: exported counts, all original job/review fields, unique IDs, linked employers and 180-day flags verified.')
        print('No selected rows removed. Human text validation is still pending.')
        print('Outputs:',OUTPUT.resolve())

finally:
    plt.show = _previous_show
_team_logs.append(_step_log.getvalue())

## 第6步｜查看样本概况


In [ ]:
import io, contextlib
import matplotlib.pyplot as plt
_step_log = io.StringIO()
_previous_show = plt.show
plt.show = lambda: plt.close('all')
try:
    with contextlib.redirect_stdout(_step_log):
        import numpy as np
        import matplotlib
        import matplotlib.pyplot as plt
        import re,json,zipfile,html
        ANALYSIS=Path('analysis_outputs')
        ANALYSIS.mkdir(exist_ok=True)
        ORDER=['Celonis','Anaplan','AlphaSense']
        COLORS={'Celonis':'#6954d8','Anaplan':'#26968f','AlphaSense':'#e6a145'}
        plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.spines.top':False,'axes.spines.right':False})
        print('Source records:',len(jobs),'postings;',len(reviews),'reviews;',len(links),'role links.')
        fig,ax=plt.subplots(figsize=(9,4.2));x=np.arange(3);w=.35
        jcounts=jobs.groupby('organization_name').size().reindex(ORDER)
        rcounts=reviews.groupby('organization_name').size().reindex(ORDER)
        ax.bar(x-w/2,jcounts,w,color='#6954d8',label='Postings')
        ax.bar(x+w/2,rcounts,w,color='#26968f',label='Reviews')
        for i in range(3):
            ax.text(i-w/2,jcounts.iloc[i]+5,str(jcounts.iloc[i]),ha='center')
            ax.text(i+w/2,rcounts.iloc[i]+5,str(rcounts.iloc[i]),ha='center')
        ax.set(xticks=x,xticklabels=ORDER,ylabel='Source records',ylim=(0,410),title='01 | Source-specific sample sizes')
        ax.legend(frameon=False);fig.tight_layout();fig.savefig(ANALYSIS/'01_sample_sizes.png',dpi=180);plt.show()
finally:
    plt.show = _previous_show
_team_logs.append(_step_log.getvalue())

counts = pd.DataFrame({
    '公司':ORDER,
    '招聘广告':[int(jobs.organization_name.eq(n).sum()) for n in ORDER],
    '源评价':[int(reviews.organization_name.eq(n).sum()) for n in ORDER],
    '初级标题广告候选':[int((jobs.organization_name.eq(n)&jobs.early_career_title_flag.fillna(False)).sum()) for n in ORDER]})
try:
    from IPython.display import display
except ImportError:
    def display(value): print(value.to_string(index=False))
display(counts)

## 第7步｜先看招聘广告：公司提到了什么？


In [ ]:
import io, contextlib
import matplotlib.pyplot as plt
_step_log = io.StringIO()
_previous_show = plt.show
plt.show = lambda: plt.close('all')
try:
    with contextlib.redirect_stdout(_step_log):
        SPECIFIC_SIGNALS={
         'Learning expressions':r'70[–-]20[–-]10|mentorship programs|mentoring by|weekly coaching|structured.{0,12}week onboarding|dedicated learning platform|learning and continuous development|training opportunities|professional development',
         'Work-arrangement expressions':r'hybrid work model|hybrid working|remote work|work from home|flexible working hours|flexible hybrid work',
         'Benefit expressions':r'paid leave|parental leave|paid time off|health insurance|medical.{0,35}coverage|retirement|401\s*\(?k\)?|tuition reimbursement|unlimited PTO'
        }
        JOB_SIGNALS={
         'Learning expressions':r'\btrain\w*\b|\bmentor\w*\b|\blearn\w*\b|\bonboard\w*\b|career development|career growth|career progression|professional development|\bcertifications\b',
         'Work-arrangement expressions':r'\bhybrid\b|\bremote\b|work from home|flexible working|flexible hours|work.life balance|work/life balance|\bovertime\b|\bshifts\b',
         'Benefit expressions':r'\bbenefits\b|\bcompensation\b|\bsalary\b|paid leave|parental leave|paid time off|health insurance|retirement|401\s*\(?k\)?|unlimited PTO'
        }
        specific_summary=[]
        for name in ORDER:
            g=jobs[jobs.organization_name.eq(name)]
            for label,pat in SPECIFIC_SIGNALS.items():
                specific_summary.append({'company':name,'specific_expression_group':label,'n':int(g.description_clean.str.contains(pat,case=False,regex=True).sum()),'n_postings':len(g)})
        pd.DataFrame(specific_summary).to_csv(ANALYSIS/'specific_phrase_diagnostics_NOT_THEME_COVERAGE.csv',index=False)
        signal_columns=[]
        for i,(label,pat) in enumerate(JOB_SIGNALS.items()):
            col=f'job_signal_{i}';signal_columns.append(col)
            jobs[col]=jobs.description_clean.str.contains(pat,case=False,regex=True)
        baseline=[]
        for name in ORDER:
            g=jobs[jobs.organization_name.eq(name)]
            for (label,pat),col in zip(JOB_SIGNALS.items(),signal_columns):
                baseline.append({'company':name,'signal':label,'n_with_expression':int(g[col].sum()),'n_postings':len(g),'pct':100*g[col].mean()})
        baseline=pd.DataFrame(baseline)
        baseline.to_csv(ANALYSIS/'job_postings_baseline_SCREENING.csv',index=False)
        print(baseline.round(1).to_string(index=False))
        fig,axes=plt.subplots(1,3,figsize=(12,4.8),sharey=True)
        for ax,label in zip(axes,JOB_SIGNALS):
            b=baseline[baseline.signal.eq(label)].set_index('company').reindex(ORDER)
            ax.bar(range(3),b.pct,color=[COLORS[n] for n in ORDER])
            for i,n in enumerate(ORDER):ax.text(i,b.loc[n,'pct']+2,f"{int(b.loc[n,'n_with_expression'])}/{int(b.loc[n,'n_postings'])}",ha='center',fontsize=9)
            ax.set(xticks=range(3),xticklabels=ORDER,ylim=(0,113),title=label)
            ax.tick_params(axis='x',rotation=25)
        axes[0].set_ylabel('% of employer postings')
        fig.suptitle('02 | Defined expressions only — not verified promises',fontweight='bold')
        fig.tight_layout();fig.savefig(ANALYSIS/'02_postings_screening.png',dpi=180);plt.show()
        early_jobs=jobs[jobs.organization_name.eq('Celonis') & jobs.early_career_title_flag.fillna(False)].copy()
        early_jobs['conversion_path_expression']=early_jobs.description_clean.str.contains(r'pathway into full.time BDR|standard path into full.time BDR',case=False,regex=True)
        early_jobs['onsite_requirement_and_hybrid_template']=early_jobs.description_clean.str.contains(r'work (?:one-site|on-site|onsite)',case=False,regex=True) & early_jobs.description_clean.str.contains('hybrid work',case=False)
        def evidence_windows(text,pattern,limit=4):
            intervals=[]
            for match in re.finditer(pattern,text,re.I):
                a=max(0,match.start()-100);b=min(len(text),match.end()+220)
                if intervals and a<=intervals[-1][1]:intervals[-1]=(intervals[-1][0],max(b,intervals[-1][1]))
                else:intervals.append((a,b))
            return ' ... '.join(text[a:b] for a,b in intervals[:limit])
        early_jobs['qualification_excerpts']=early_jobs.description_clean.map(lambda t:evidence_windows(t,r'student|enrolled|graduate|months|proficien|fluent|degree',5))
        early_jobs['support_excerpts']=early_jobs.description_clean.map(lambda t:evidence_windows(t,r'mentorship|mentoring|coaching|70.20.10|onboarding|pathway into|applicable regions',5))
        early_jobs.to_csv(ANALYSIS/'Celonis_18_early_title_jobs_evidence.csv',index=False,encoding='utf-8-sig')
        print('Celonis early-title postings:',len(early_jobs))
        print('BDR pathway expression:',int(early_jobs.conversion_path_expression.sum()))
        print('Onsite role wording plus hybrid template:',int(early_jobs.onsite_requirement_and_hybrid_template.sum()),'(scope clarification candidates, not proven contradictions)')
finally:
    plt.show = _previous_show
_team_logs.append(_step_log.getvalue())

_img = plt.imread(ANALYSIS/'02_postings_screening.png')
_fig, _ax = plt.subplots(figsize=(12,12*_img.shape[0]/_img.shape[1]))
_ax.imshow(_img); _ax.axis('off'); _fig.tight_layout(pad=0); plt.show()

### 加入评价前，先固定广告阶段的初步判断
## 第8步｜加入评价前，检查面试语境


In [ ]:
import io, contextlib
import matplotlib.pyplot as plt
_step_log = io.StringIO()
_previous_show = plt.show
plt.show = lambda: plt.close('all')
try:
    with contextlib.redirect_stdout(_step_log):
        context_decisions = {}
        decisions = []
        for rid, (decision, reason) in context_decisions.items():
            row = reviews[reviews.review_id.eq(rid)].iloc[0]
            decisions.append({'review_id': rid, 'company': row.organization_name, 'decision': decision, 'reason': reason, 'reviewer': 'ChatGPT-assisted context check; human review pending'})
        context_table = pd.DataFrame(decisions)
        context_table.to_csv(ANALYSIS / 'interview_context_decisions_AI_ASSISTED.csv', index=False)
        exclude_ids = {rid for rid, (d, _) in context_decisions.items() if d == 'exclude_interview_only'}
        work_reviews = reviews[~reviews.review_id.isin(exclude_ids)].copy()
        print('Retained source reviews:', len(reviews), '; provisional workplace-focused sample:', len(work_reviews))
        print(context_table.to_string(index=False))
finally:
    plt.show = _previous_show
_team_logs.append(_step_log.getvalue())


## 第9步｜计算员工体验评分


In [ ]:
import io, contextlib
import matplotlib.pyplot as plt
_step_log = io.StringIO()
_previous_show = plt.show
plt.show = lambda: plt.close('all')
try:
    with contextlib.redirect_stdout(_step_log):
        DIMS={'career_opportunities_valid':'Career opportunities','work_life_balance_valid':'Work-life balance','compensation_and_benefits_valid':'Compensation & benefits'}
        def score_table(df,label):
            rows=[]
            for name in ORDER:
                group=df[df.organization_name.eq(name)]
                for col,title in DIMS.items():
                    v=pd.to_numeric(group[col],errors='coerce').dropna()
                    rows.append({'sample':label,'company':name,'dimension':title,'mean':v.mean(),'n_valid':len(v),'n_source_reviews':len(group)})
            return pd.DataFrame(rows)
        scores=score_table(work_reviews,'workplace_context_screened')
        full_scores=score_table(reviews,'all_source_reviews')
        pd.concat([scores,full_scores]).to_csv(ANALYSIS/'review_dimension_scores.csv',index=False)
        print(scores.round(3).to_string(index=False))
        fig,axes=plt.subplots(1,3,figsize=(12,4.8),sharey=True)
        for ax,title in zip(axes,DIMS.values()):
            s=scores[scores.dimension.eq(title)].set_index('company').reindex(ORDER)
            for i,name in enumerate(ORDER):
                ax.scatter(i,s.loc[name,'mean'],s=85,color=COLORS[name])
                ax.text(i,s.loc[name,'mean']+.13,f"{s.loc[name,'mean']:.2f}\nn={int(s.loc[name,'n_valid'])}",ha='center',fontsize=9)
            ax.set(xticks=range(3),xticklabels=ORDER,xlim=(-.6,2.6),ylim=(1,5.1),title=title)
            ax.tick_params(axis='x',rotation=25);ax.grid(axis='y',alpha=.15)
        axes[0].set_ylabel('Mean valid 1–5 rating; 0 excluded')
        fig.suptitle('03 | Reported experiences — descriptive, unmatched samples',fontweight='bold')
        fig.tight_layout();fig.savefig(ANALYSIS/'03_review_dimensions.png',dpi=180);plt.show()

        REVIEW_THEMES={
         'Career':r'\blearn\w*\b|\btrain\w*\b|\bmentor\w*\b|\bpromot\w*\b|\bcareer\b|\bgrow\w*\b|\bprogression\b',
         'Work arrangements / workload':r'work[ -]?life|\bwlb\b|\bovertime\b|\bworkload\b|\bhours\b|\bburnout\b|\bremote\b|\bhybrid\b|\bflexib\w*\b|\bpressure\b|\bstress\w*\b',
         'Pay / benefits':r'\bpay\b|\bsalary\b|\bsalaries\b|\bcompensation\b|\bbenefit\w*\b|\bbonus\w*\b|\binsurance\b|\braise\w*\b|\bhike\w*\b|\bpension\b'}
        mentions=[]
        for name in ORDER:
            g=work_reviews[work_reviews.organization_name.eq(name)]
            for theme,pat in REVIEW_THEMES.items():
                for field in ['pros_clean','cons_clean']:
                    count=int(g[field].str.contains(pat,case=False,regex=True).sum())
                    mentions.append({'company':name,'theme':theme,'source_field':field,'n_mentions':count,'n_reviews':len(g),'pct':100*count/len(g)})
        mentions=pd.DataFrame(mentions)
        mentions.to_csv(ANALYSIS/'review_field_theme_mentions_SCREENING.csv',index=False)
        fig,axes=plt.subplots(1,3,figsize=(12,4.8),sharey=True)
        for ax,theme in zip(axes,REVIEW_THEMES):
            g=mentions[mentions.theme.eq(theme)].pivot(index='company',columns='source_field',values='pct').reindex(ORDER)
            ax.bar(x-.17,g.pros_clean,.34,color='#26968f',label='Pros field')
            ax.bar(x+.17,g.cons_clean,.34,color='#d68163',label='Cons field')
            ax.set(xticks=x,xticklabels=ORDER,title=theme,ylim=(0,40))
            ax.tick_params(axis='x',rotation=25)
        axes[0].set_ylabel('% of workplace-focused review records')
        axes[0].legend(frameon=False)
        fig.suptitle('04 | Theme mentions by source field — not sentiment labels',fontweight='bold')
        fig.tight_layout();fig.savefig(ANALYSIS/'04_review_theme_mentions.png',dpi=180);plt.show()
finally:
    plt.show = _previous_show
_team_logs.append(_step_log.getvalue())

_img = plt.imread(ANALYSIS/'03_review_dimensions.png')
_fig, _ax = plt.subplots(figsize=(12,12*_img.shape[0]/_img.shape[1]))
_ax.imshow(_img); _ax.axis('off'); _fig.tight_layout(pad=0); plt.show()

## 第10步｜换样本范围，检查结论是否稳定


In [ ]:
import io, contextlib
import matplotlib.pyplot as plt
_step_log = io.StringIO()
_previous_show = plt.show
plt.show = lambda: plt.close('all')
try:
    with contextlib.redirect_stdout(_step_log):
        role_patterns={'Sales development':r'\bBDR\b|\bSDR\b|sales development|business development representative',
         'Software engineering':r'software engineer|software developer'}
        review_dates=pd.to_datetime(work_reviews.review_date,errors='coerce')
        common_start=work_reviews.groupby('organization_name').review_date.min().max()
        common_end=work_reviews.groupby('organization_name').review_date.max().min()
        subsets={'All source reviews':reviews,'Context-screened':work_reviews,
         '2025 onward':work_reviews[review_dates.ge(pd.Timestamp('2025-01-01'))],
         'Common date window':work_reviews[work_reviews.review_date.between(common_start,common_end)],
         'Early-title reviews':work_reviews[work_reviews.early_career_review_title_flag.fillna(False)]}
        for label,pat in role_patterns.items():subsets[label]=work_reviews[work_reviews.reviewer_job_title_clean.str.contains(pat,case=False,regex=True)]
        for label,mask in [('Linked same-role',pd.Series(True,index=links.index)),('Linked within 180 days',links.within_180_days_flag.fillna(False)),('Linked exact location',links.exact_location_text_flag)]:
            subsets[label]=work_reviews[work_reviews.review_id.isin(set(links.loc[mask,'review_id']))]
        sensitivity=pd.concat([score_table(df,label) for label,df in subsets.items()],ignore_index=True)
        sensitivity.to_csv(ANALYSIS/'score_sensitivity_by_time_role_and_link.csv',index=False)
        job_unique=jobs.drop_duplicates(['organization_id','text_sha256'])
        job_unique.groupby('organization_name')[signal_columns].mean().mul(100).to_csv(ANALYSIS/'job_unique_description_sensitivity_pct.csv')
        print('Common review window:',common_start,'to',common_end)
        print('Strict exact-location linked reviews:',len(subsets['Linked exact location']))
        print('Early-title subset counts:',subsets['Early-title reviews'].groupby('organization_name').size().to_dict())
        fig,ax=plt.subplots(figsize=(10,4.8))
        focus=sensitivity[sensitivity.company.eq('Celonis') & sensitivity['sample'].isin(['All source reviews','Context-screened','2025 onward','Early-title reviews'])]
        offsets=np.linspace(-.22,.22,4);palette=['#9ba4b4','#6954d8','#26968f','#d68163']
        for offset,(label,color) in zip(offsets,zip(['All source reviews','Context-screened','2025 onward','Early-title reviews'],palette)):
            g=focus[focus['sample'].eq(label)].set_index('dimension').reindex(list(DIMS.values()))
            ax.scatter(np.arange(3)+offset,g['mean'],s=55,label=label,color=color)
            for i,row in enumerate(g.itertuples()):ax.text(i+offset,row.mean+.06,f'n={row.n_valid}',ha='center',fontsize=7,rotation=45)
        ax.set(xticks=range(3),xticklabels=list(DIMS.values()),ylabel='Mean valid 1–5 rating',ylim=(1,5.2),xlim=(-.6,2.6),title='05 | Celonis: check sensitivity before interpreting differences')
        ax.legend(frameon=False,loc='lower left',ncol=2);ax.grid(axis='y',alpha=.15)
        fig.tight_layout();fig.savefig(ANALYSIS/'05_celonis_sensitivity.png',dpi=180);plt.show()
finally:
    plt.show = _previous_show
_team_logs.append(_step_log.getvalue())

_img = plt.imread(ANALYSIS/'05_celonis_sensitivity.png')
_fig, _ax = plt.subplots(figsize=(12,12*_img.shape[0]/_img.shape[1]))
_ax.imshow(_img); _ax.axis('off'); _fig.tight_layout(pad=0); plt.show()

## 第11步｜保留证据，完成人工核查后再定稿


In [ ]:
import io, contextlib
import matplotlib.pyplot as plt
_step_log = io.StringIO()
_previous_show = plt.show
plt.show = lambda: plt.close('all')
try:
    with contextlib.redirect_stdout(_step_log):
        evidence_specs = []
        evidence = []
        for source, rid, theme, interpretation in evidence_specs:
            df = jobs if source == 'job' else reviews
            idcol = 'job_id' if source == 'job' else 'review_id'
            row = df[df[idcol].eq(rid)].iloc[0]
            evidence.append({'source': source, 'source_id': rid, 'company': row.organization_name, 'title': row.title if source == 'job' else row.reviewer_job_title, 'date': row.posted_at if source == 'job' else row.review_date, 'theme': theme, 'interpretation': interpretation, 'full_text': row.description_clean if source == 'job' else row.review_text_clean, 'checked_by': 'ChatGPT-assisted evidence reading; human check pending'})
        evidence = pd.DataFrame(evidence, columns=['source','source_id','company','theme','interpretation','checked_by'])
        evidence.to_csv(ANALYSIS / 'evidence_register_AI_ASSISTED.csv', index=False, encoding='utf-8-sig')
        print(evidence[['source', 'source_id', 'company', 'theme', 'interpretation']].to_string(index=False))
        audit_path = ANALYSIS / 'human_audit_30.csv'
        if not audit_path.exists():
            audit = []
            for source, df, idcol in [('job', jobs, 'job_id'), ('review', work_reviews, 'review_id')]:
                for name in ORDER:
                    group = df[df.organization_name.eq(name)]
                    if name == 'Celonis':
                        flag = 'early_career_title_flag' if source == 'job' else 'early_career_review_title_flag'
                        group = group[group[flag].fillna(False)]
                    for _, row in group.sample(5, random_state=20261023).iterrows():
                        rec = {'source': source, 'source_id': row[idcol], 'company': name, 'title': row.title if source == 'job' else row.reviewer_job_title, 'full_text': row.description_clean if source == 'job' else row.review_text_clean, 'human_checked': '', 'human_reviewer': '', 'notes': ''}
                        pats = list(JOB_SIGNALS.values()) if source == 'job' else list(REVIEW_THEMES.values())
                        for label, pat in zip(['career', 'work', 'benefits'], pats):
                            rec[label + '_candidate'] = int(bool(re.search(pat, rec['full_text'], re.I)))
                            rec[label + '_human'] = ''
                        audit.append(rec)
            pd.DataFrame(audit).to_csv(audit_path, index=False, encoding='utf-8-sig')
        audit = pd.read_csv(audit_path, dtype=str, keep_default_na=False)
        completed = audit.human_checked.str.lower().isin(['yes', 'true', '1'])
        print('Human-checked:', int(completed.sum()), '/30')
        metrics = []
        for source in ['job', 'review']:
            for theme in ['career', 'work', 'benefits']:
                g = audit[audit.source.eq(source) & completed & audit[theme + '_human'].isin(['0', '1'])]
                if g.empty:
                    continue
                y = g[theme + '_human'].astype(int)
                p = g[theme + '_candidate'].astype(int)
                tp = int(((p == 1) & (y == 1)).sum())
                fp = int(((p == 1) & (y == 0)).sum())
                fn = int(((p == 0) & (y == 1)).sum())
                metrics.append({'source': source, 'theme': theme, 'n_checked': len(g), 'agreement': float((p == y).mean()), 'precision': tp / (tp + fp) if tp + fp else np.nan, 'recall': tp / (tp + fn) if tp + fn else np.nan})
        pd.DataFrame(metrics).to_csv(ANALYSIS / 'human_validation_metrics.csv', index=False)
        if not metrics:
            print('No classification performance claimed: human annotations pending.')
        analysis_manifest = {'focal': 'Celonis', 'peers': ['Anaplan', 'AlphaSense'], 'source_counts': {'jobs': len(jobs), 'reviews': len(reviews), 'links': len(links)}, 'workplace_context_screened_reviews': len(work_reviews), 'provisional_excluded_review_ids': sorted(exclude_ids), 'rules': JOB_SIGNALS, 'specific_phrase_rules': SPECIFIC_SIGNALS, 'review_theme_rules': REVIEW_THEMES, 'human_validation_status': 'Completed sample; see per-theme labelled counts' if int(completed.sum()) == 30 else 'Pending / partial', 'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'matplotlib': matplotlib.__version__}
        (ANALYSIS / 'analysis_manifest.json').write_text(json.dumps(analysis_manifest, ensure_ascii=False, indent=2))
finally:
    plt.show = _previous_show
_team_logs.append(_step_log.getvalue())


### 两项初步建议：证据、限制和下一步测量
## 第12步｜只下载一个结果包


In [ ]:
from pathlib import Path
import zipfile
(ANALYSIS/'run_log.txt').write_text('\n'.join(_team_logs),encoding='utf-8')
readme = """Celonis团队分析结果
研究问题：Celonis如何通过改善工作体验和招聘信息，更好地吸引初入职场的年轻人？
core_charts：3张核心图。
inputs：相同的5份输入数据；preprocessing：清洗表、字典、缺失与异常日志。
details：详细统计、完整5图、证据原文、语境判断和人工核查表。
重新运行：在Colab上传并运行配套Celonis_Team_Step_by_Step.ipynb，无需另外上传CSV。
注意：关键词提及不是承诺或情绪；评分未匹配，不做企业排名。人工核查进度以human_audit_30.csv和analysis_manifest.json为准。
数据无招聘/留任结果，不进行因果或效果预测。AI辅助不算人工核查。最终PPT、审核、排练和提交仍由团队完成。
"""
result_zip = Path('Celonis_Team_Results.zip')
with zipfile.ZipFile(result_zip,'w',zipfile.ZIP_DEFLATED) as z:
    z.writestr('READ_ME.txt',readme)
    for folder,archive_folder in [(DATA,'inputs'),(OUTPUT,'preprocessing'),(ANALYSIS,'details')]:
        for file in sorted(folder.glob('*')):
            if file.is_file(): z.write(file,str(Path(archive_folder)/file.name))
    for name in ['02_postings_screening.png','03_review_dimensions.png','05_celonis_sensitivity.png']:
        z.write(ANALYSIS/name,str(Path('core_charts')/name))
print('完成：Celonis_Team_Results.zip。人工核查进度：',int(completed.sum()),'/30。')
print('在左侧Files → Celonis_Team_Workspace下载这一个ZIP即可。')